# Philippine Address Pipeline

Process: Add raw data -> normalize with `ph_area_dictionary` and `ph_address_alias` -> complete abbreviated or aliased addresses -> load to OSM Nominatim -> map the returned place back to `dim_location` codes.

## Cell 0 — Environment Setup

Check all required packages are installed (pandas, numpy, rapidfuzz, tqdm, openpyxl, xlsxwriter).

In [342]:
import importlib, sys

REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit paths and thresholds: `CITY_SCORE_HIGH` (88), `CITY_SCORE_LOW` (65), `PROV_SCORE_HIGH` (88), `PROV_SCORE_LOW` (60), `MAX_ROWS` (None for all rows).

In [343]:
from pathlib import Path

# -- File paths --
BASE_DIR = Path("..") / ".." / "data"   # draft notebook is in update_address/notebooks/drafts

INPUT_FILE   = str(BASE_DIR / "sample" / "copy_ps_address_v3.xlsx")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
AREA_DICT_FILE = str(BASE_DIR / "dictionary" / "ph_area_dictionary_v2.json")
OUT_VERIFIED = str(BASE_DIR / "output"  / "verified_addresses.xlsx")
OUT_INVALID  = str(BASE_DIR / "output"  / "invalid_addresses.xlsx")

# -- OSM Nominatim settings -----------------------------------------------------
USE_API = True
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
NOMINATIM_USER_AGENT = "update_address_pipeline/1.0"
NOMINATIM_EMAIL = ""
NOMINATIM_TIMEOUT = 20
NOMINATIM_PAUSE_SECONDS = 1.0
API_QUERY_MODE = "freeform"

# -- Batch input (unchanged) --------------------------------------------------
input_paths = [
    # Adjust range to cover your batches, e.g. range(1, 101)
]

# -- Fuzzy-match thresholds (0-100) ------------------------------------------
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# -- Run controls -------------------------------------------------------------
MAX_ROWS       = 10000          # None = all rows; set e.g. 100 for quick test runs

# -- Quick path check ---------------------------------------------------------
if input_paths:
    print(f'  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = 'OK' if p.exists() else 'NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = 'OK' if Path(INPUT_FILE).exists() else 'NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE, AREA_DICT_FILE]:
    status = 'OK' if Path(f).exists() else 'NOT FOUND'
    print(f'  {status}  {f}')


  OK  ..\..\data\sample\copy_ps_address_v3.xlsx
  OK  ..\..\data\mapping\dim_location_20260316_v2.csv
  OK  ..\..\data\utils\ph_address_alias_extended_v4.csv
  OK  ..\..\data\dictionary\ph_area_dictionary_v2.json


## Cell 2 — Imports & Logging

Load libraries (pandas, rapidfuzz, tqdm) and configure logging with timestamps.

In [344]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


11:26:06  INFO      Imports OK


## Cell 3 — Add Raw Data and Reference Files

Load `dim_location`, `ph_address_alias`, and `ph_area_dictionary`. These files provide the raw PSGC hierarchy, alias rules, and dictionary context used by the normalizer.

In [345]:
def _read_csv_with_fallback(path: str) -> pd.DataFrame:
    strict_encodings = ['utf-8', 'utf-8-sig']
    for enc in strict_encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    # Keep mostly-UTF8 files readable even if they contain a few broken bytes.
    try:
        log.warning(f'Loaded {path} using utf-8 with replacement for invalid bytes')
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    except Exception:
        pass

    for enc in ['cp1252', 'latin1']:
        try:
            log.warning(f'Loaded {path} using fallback encoding: {enc}')
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f'Unable to decode CSV file: {path}')


def _repair_mojibake_geo(text: str) -> str:
    t = str(text).strip()
    if any(ch in t for ch in ('Ã', 'Â', '�')):
        try:
            t = t.encode('latin1').decode('utf-8')
        except Exception:
            pass
    t = t.replace('�', 'Ñ')
    t = t.upper()
    t = re.sub(r'\bPARA\s*AQUE\b', 'PARANAQUE', t)
    t = re.sub(r'\bLAS\s*PI\s*AS\b', 'LAS PINAS', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = _read_csv_with_fallback(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).apply(_repair_mojibake_geo)

    log.info('Loading alias table ...')
    alias = _read_csv_with_fallback(alias_path)
    alias['pattern'] = alias['pattern'].fillna('').astype(str).apply(_repair_mojibake_geo)
    alias['replacement'] = alias['replacement'].fillna('').astype(str).apply(_repair_mojibake_geo)

    cities = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n-- Sample dim_location --')
display(dim.head(3))
print('\n-- Sample aliases --')
display(alias_df.head(10))


11:26:06  INFO      Loading dim_location ...
11:26:07  INFO      Loading alias table ...
11:26:07  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



-- Sample dim_location --


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



-- Sample aliases --


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


In [346]:
# Dictionary-backed lookup tables used by the strict matcher.

def _canon_dict_text(text: str) -> str:
    import unicodedata
    value = unicodedata.normalize('NFKD', str(text))
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    value = value.upper()
    value = re.sub(r'[^A-Z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()


def load_area_dictionary(dict_path: str):
    log.info('Loading ph_area_dictionary ...')
    with open(dict_path, 'r', encoding='utf-8') as f:
        area_dict = json.load(f)

    dict_province_alias_map = {}
    dict_province_display_map = {}
    dict_city_alias_map = {}
    dict_city_display_map = {}
    dict_city_province_map = {}
    dict_city_region_map = {}
    dict_area_term_to_entries = {}
    dict_street_like_terms = set()
    dict_anchor_tokens = set()
    dict_text_replacements = {}

    for term in area_dict.get('street_like_terms', []):
        term_can = _canon_dict_text(term)
        if term_can:
            dict_street_like_terms.add(term_can)

    global_rules = area_dict.get('global_rules', {}) if isinstance(area_dict, dict) else {}
    for token in global_rules.get('anchor_tokens', []) or []:
        token_can = _canon_dict_text(token)
        if token_can:
            dict_anchor_tokens.add(token_can)

    dict_require_anchor_for_city_inference = bool(global_rules.get('require_anchor_for_city_inference', True))
    dict_reject_if_only_street_like = bool(global_rules.get('reject_if_only_street_like', True))

    for province_name, aliases in (area_dict.get('province_aliases', {}) or {}).items():
        province_can = _canon_dict_text(province_name)
        if not province_can:
            continue
        dict_province_alias_map[province_can] = province_can
        dict_province_display_map[province_can] = province_name
        dict_text_replacements[province_can] = province_name
        for alias in aliases or []:
            alias_can = _canon_dict_text(alias)
            if alias_can:
                dict_province_alias_map[alias_can] = province_can
                dict_text_replacements[alias_can] = province_name

    for city_key, city_meta in (area_dict.get('cities', {}) or {}).items():
        if not isinstance(city_meta, dict):
            continue

        city_can = _canon_dict_text(city_key)
        if not city_can:
            continue

        dict_city_alias_map[city_can] = city_can
        dict_city_display_map[city_can] = city_key
        dict_text_replacements[city_can] = city_key

        province_can = _canon_dict_text(city_meta.get('province', ''))
        province_can = dict_province_alias_map.get(province_can, province_can)
        if province_can:
            dict_province_alias_map.setdefault(province_can, province_can)
        dict_city_province_map[city_can] = province_can or None

        region_can = _canon_dict_text(city_meta.get('region', ''))
        dict_city_region_map[city_can] = region_can or None

        for alias in city_meta.get('aliases', []) or []:
            alias_can = _canon_dict_text(alias)
            if alias_can:
                dict_city_alias_map[alias_can] = city_can
                dict_text_replacements[alias_can] = city_key

        for area_name, area_meta in (city_meta.get('areas', {}) or {}).items():
            if not isinstance(area_meta, dict):
                continue

            area_can = _canon_dict_text(area_name)
            if not area_can:
                continue

            dict_text_replacements[area_can] = area_name
            entry = {
                'city_can': city_can,
                'infer_allowed': bool(area_meta.get('infer_city_allowed', False)),
                'requires_context': bool(area_meta.get('requires_context', False)),
                'priority': int(area_meta.get('priority', 0)),
                'area_type': str(area_meta.get('type', 'area')),
            }
            dict_area_term_to_entries.setdefault(area_can, []).append(entry)

            for alias in area_meta.get('aliases', []) or []:
                alias_can = _canon_dict_text(alias)
                if alias_can:
                    dict_area_term_to_entries.setdefault(alias_can, []).append(entry.copy())
                    dict_text_replacements[alias_can] = area_name

    # Backfill city aliases from province_city_map so municipalities not present
    # under area_dict['cities'] are still detectable as explicit city signals.
    for prov_name, city_list in (area_dict.get('province_city_map', {}) or {}).items():
        prov_can = _canon_dict_text(prov_name)
        prov_can = dict_province_alias_map.get(prov_can, prov_can)
        for city_name in city_list or []:
            city_can = _canon_dict_text(city_name)
            if not city_can:
                continue

            dict_city_alias_map.setdefault(city_can, city_can)
            dict_city_display_map.setdefault(city_can, city_name)
            dict_text_replacements.setdefault(city_can, city_name)

            if prov_can:
                dict_city_province_map.setdefault(city_can, prov_can)

    # Backfill any remaining standalone city names from all_cities.
    for city_name in (area_dict.get('all_cities', []) or []):
        city_can = _canon_dict_text(city_name)
        if not city_can:
            continue
        dict_city_alias_map.setdefault(city_can, city_can)
        dict_city_display_map.setdefault(city_can, city_name)
        dict_text_replacements.setdefault(city_can, city_name)

    dict_city_terms = sorted(dict_city_alias_map.keys(), key=len, reverse=True)
    dict_province_terms = sorted(dict_province_alias_map.keys(), key=len, reverse=True)
    dict_area_terms = sorted(dict_area_term_to_entries.keys(), key=len, reverse=True)
    dict_single_word_city_allow = {term for term in dict_city_terms if len(term.split()) == 1}

    dict_replacement_terms = sorted(dict_text_replacements.keys(), key=len, reverse=True)
    dict_replacement_pattern = None
    if dict_replacement_terms:
        dict_replacement_pattern = re.compile(r'\b(' + '|'.join(re.escape(t) for t in dict_replacement_terms) + r')\b')

    log.info(
        f'ph_area_dictionary: {len(dict_city_display_map):,} cities | '
        f'{len(dict_province_display_map):,} provinces | '
        f'{len(dict_area_terms):,} area terms'
    )

    return {
        'area_dict': area_dict,
        'dict_city_alias_map': dict_city_alias_map,
        'dict_city_display_map': dict_city_display_map,
        'dict_city_province_map': dict_city_province_map,
        'dict_city_region_map': dict_city_region_map,
        'dict_city_terms': dict_city_terms,
        'dict_province_alias_map': dict_province_alias_map,
        'dict_province_display_map': dict_province_display_map,
        'dict_province_terms': dict_province_terms,
        'dict_area_term_to_entries': dict_area_term_to_entries,
        'dict_area_terms': dict_area_terms,
        'dict_single_word_city_allow': dict_single_word_city_allow,
        'dict_street_like_terms': dict_street_like_terms,
        'dict_anchor_tokens': dict_anchor_tokens,
        'dict_require_anchor_for_city_inference': dict_require_anchor_for_city_inference,
        'dict_reject_if_only_street_like': dict_reject_if_only_street_like,
        'dict_text_replacements': dict_text_replacements,
        'dict_replacement_pattern': dict_replacement_pattern,
    }


area_dict_ctx = load_area_dictionary(AREA_DICT_FILE)
area_dict = area_dict_ctx['area_dict']
dict_city_alias_map = area_dict_ctx['dict_city_alias_map']
dict_city_display_map = area_dict_ctx['dict_city_display_map']
dict_city_province_map = area_dict_ctx['dict_city_province_map']
dict_city_region_map = area_dict_ctx['dict_city_region_map']
dict_city_terms = area_dict_ctx['dict_city_terms']
dict_province_alias_map = area_dict_ctx['dict_province_alias_map']
dict_province_display_map = area_dict_ctx['dict_province_display_map']
dict_province_terms = area_dict_ctx['dict_province_terms']
dict_area_term_to_entries = area_dict_ctx['dict_area_term_to_entries']
dict_area_terms = area_dict_ctx['dict_area_terms']
dict_single_word_city_allow = area_dict_ctx['dict_single_word_city_allow']
dict_street_like_terms = area_dict_ctx['dict_street_like_terms']
dict_anchor_tokens = area_dict_ctx['dict_anchor_tokens']
dict_require_anchor_for_city_inference = area_dict_ctx['dict_require_anchor_for_city_inference']
dict_reject_if_only_street_like = area_dict_ctx['dict_reject_if_only_street_like']
dict_text_replacements = area_dict_ctx['dict_text_replacements']
dict_replacement_pattern = area_dict_ctx['dict_replacement_pattern']

print('\n-- Dictionary summary --')
print(f"Cities   : {len(dict_city_display_map):,}")
print(f"Provinces: {len(dict_province_display_map):,}")
print(f"Areas    : {len(dict_area_terms):,}")
print(f"Anchors  : {sorted(dict_anchor_tokens)}")

11:26:07  INFO      Loading ph_area_dictionary ...
11:26:07  INFO      ph_area_dictionary: 1,436 cities | 18 provinces | 157 area terms



-- Dictionary summary --
Cities   : 1,436
Provinces: 18
Areas    : 157
Anchors  : ['BARANGAY', 'BGY', 'BRGY', 'CITY', 'MUN', 'MUNICIPALITY', 'PROV', 'PROVINCE']


## Cell 4 — Stage 1: Alias Normalization

Build compiled regex from 499 alias pairs (sorted longest-first) to normalize address text. Remove duplicate tokens like "CITY CITY" after transformation.

In [347]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip() and isinstance(r, str) and r.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest pattern first
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = _repair_mojibake_geo(text)

    # Normalize punctuation before dictionary replacements to align token boundaries.
    text = re.sub(r'[\.,;:/\\\-\(\)\[\]\{\}]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Apply dictionary.json aliases first (cities, provinces, areas, acronyms).
    dict_repl = globals().get('dict_text_replacements', {})
    dict_pat = globals().get('dict_replacement_pattern')
    if dict_repl and dict_pat:
        text = dict_pat.sub(lambda m: dict_repl.get(m.group(0), m.group(0)), text)

    # Keep targeted legacy normalizations that are not covered by alias tables.
    text = re.sub(r'\bC\.?\s*N\.?\b', 'CAMARINES NORTE', text)
    text = re.sub(r'\bCAINTA\s+RIZAL\b', 'CAINTA CITY RIZAL', text)
    text = re.sub(r'\bCAINTA\b', 'CAINTA CITY', text)
    text = re.sub(r'\bTANZA\b', 'TANZA CITY', text)
    text = re.sub(r'\bPQUE\b', 'PARANAQUE', text)
    text = re.sub(r'\bLASPINAS\b', 'LAS PINAS', text)
    text = re.sub(r'\bQC\b', 'QUEZON CITY', text)
    text = re.sub(r'\bMKT\b', 'MAKATI', text)
    text = re.sub(r'\bPARA\s*AQUE\b', 'PARANAQUE', text)
    text = re.sub(r'\bLAS\s*PI\s*AS\b', 'LAS PINAS', text)

    # Then apply the curated alias csv replacements.
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)

    # Remove duplicate consecutive words (e.g. 'CITY CITY', 'BARANGAY BARANGAY')
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

# -- Quick test on sample addresses --------------------------------------------
test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
    'LIONS PARK RES SUNVALLEY PQUE.',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('-' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')

11:26:07  INFO      Alias regex built from 514 patterns


ORIGINAL                                                 NORMALIZED
--------------------------------------------------------------------------------------------------------------
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY CAPITAL MISAMIS ORIENTAL
LIONS PARK RES SUNVALLEY PQUE.                           LIONS PARK RES SUN VALLEY CITY OF PARA?AQUE


## Cell 4 — Normalize and Complete Address Text

Normalize raw address text using the dictionary context plus the alias table, then expand abbreviated, acronym-based, or aliased place names into a fuller address form.

In [348]:
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.WRatio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Return (best_match_string, score) or (None, 0) if below cutoff."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Try each token against choices; return best (match, score)."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Quick smoke test ──────────────────────────────────────────────────────────
tests = [
    ('QUEZON CITY', cities),
    ('ILOCOS SUR', provinces),
    ('TAGUIG', cities),
    ('SOUTH SIGNAL', cities),   # should score low / return nothing meaningful
]
print(f'{"Query":<25}  {"Best match":<35}  Score')
print('─' * 75)
for q, lst in tests:
    m, s = best_match(q, lst, score_cutoff=0)
    print(f'{q:<25}  {str(m):<35}  {s}')


Query                      Best match                           Score
───────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                          100
ILOCOS SUR                 ILOCOS SUR                           100
TAGUIG                     CITY OF TAGUIG                       90
SOUTH SIGNAL               SIGAY                                72


## Cell 5 — Stage 2: Candidate Matching Helpers

Use helper functions to compare normalized addresses against canonical city, province, region, and barangay values before the API lookup.

In [349]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
        city_required_pass=False, city_signal=None, invalid_reason=None,
    )

    def _ascii_fold(text: str) -> str:
        import unicodedata
        txt = unicodedata.normalize('NFKD', str(text))
        return ''.join(ch for ch in txt if not unicodedata.combining(ch))

    def _repair_mojibake_geo(text: str) -> str:
        t = str(text).upper()
        t = t.replace('�', 'N')
        t = re.sub(r'\bPARA\s*AQUE\b', 'PARANAQUE', t)
        t = re.sub(r'\bLAS\s*PI\s*AS\b', 'LAS PINAS', t)
        return re.sub(r'\s+', ' ', t).strip()

    def _canon(text: str) -> str:
        text = _repair_mojibake_geo(_ascii_fold(str(text).upper()))
        text = re.sub(r'[^A-Z0-9]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    if not hasattr(match_address, '_cache'):
        dim_work = dim.copy()
        for c in ['region_name', 'province_name', 'city_name', 'barangay_name']:
            dim_work[c] = dim_work[c].fillna('').astype(str).str.upper().str.strip()
        dim_work['_city_canon'] = dim_work['city_name'].apply(_canon)
        dim_work['_prov_canon'] = dim_work['province_name'].apply(_canon)
        dim_work['_brgy_canon'] = dim_work['barangay_name'].apply(_canon)

        dim_city_variant_map = {}
        dim_city_raw_map = {}
        for city_name in sorted(x for x in dim_work['city_name'].unique().tolist() if x):
            base = _canon(city_name)
            dim_city_raw_map[base] = city_name
            variants = {base}
            if base.startswith('CITY OF '):
                core = base[len('CITY OF '):].strip()
                if core:
                    variants.add(core)
                    variants.add(f'{core} CITY')
            if base.startswith('MUNICIPALITY OF '):
                core = base[len('MUNICIPALITY OF '):].strip()
                if core:
                    variants.add(core)
            if base.endswith(' CITY'):
                core = base[:-len(' CITY')].strip()
                if core:
                    variants.add(core)
                    variants.add(f'CITY OF {core}')
            for v in variants:
                if v and v not in dim_city_variant_map:
                    dim_city_variant_map[v] = base

        barangay_values = sorted(x for x in dim_work['_brgy_canon'].unique().tolist() if x)
        barangay_set = set(barangay_values)
        brgy_token_max = max((len(x.split()) for x in barangay_values), default=1)

        barangay_to_cityset = {}
        for _, r in dim_work[['_brgy_canon', '_city_canon']].drop_duplicates().iterrows():
            b = r['_brgy_canon']
            c = r['_city_canon']
            if b and c:
                barangay_to_cityset.setdefault(b, set()).add(c)

        dict_city_alias_map = globals().get('dict_city_alias_map', {})
        dict_city_display_map = globals().get('dict_city_display_map', {})
        dict_city_province_map = globals().get('dict_city_province_map', {})
        dict_city_region_map = globals().get('dict_city_region_map', {})
        dict_city_terms = globals().get('dict_city_terms', [])
        dict_province_alias_map = globals().get('dict_province_alias_map', {})
        dict_province_terms = globals().get('dict_province_terms', [])
        dict_area_term_to_entries = globals().get('dict_area_term_to_entries', {})
        dict_area_terms = globals().get('dict_area_terms', [])
        dict_single_word_city_allow = globals().get('dict_single_word_city_allow', set())
        dict_street_like_terms = globals().get('dict_street_like_terms', set())
        dict_anchor_tokens = globals().get('dict_anchor_tokens', set())
        dict_require_anchor_for_city_inference = bool(globals().get('dict_require_anchor_for_city_inference', True))
        dict_reject_if_only_street_like = bool(globals().get('dict_reject_if_only_street_like', True))

        match_address._cache = {
            'dim_work': dim_work,
            'dim_city_variant_map': dim_city_variant_map,
            'dim_city_raw_map': dim_city_raw_map,
            'barangay_set': barangay_set,
            'brgy_token_max': brgy_token_max,
            'barangay_to_cityset': barangay_to_cityset,
            'dict_city_alias_map': dict_city_alias_map,
            'dict_city_display_map': dict_city_display_map,
            'dict_city_province_map': dict_city_province_map,
            'dict_city_region_map': dict_city_region_map,
            'dict_city_terms': dict_city_terms,
            'dict_province_alias_map': dict_province_alias_map,
            'dict_province_terms': dict_province_terms,
            'dict_area_term_to_entries': dict_area_term_to_entries,
            'dict_area_terms': dict_area_terms,
            'dict_single_word_city_allow': dict_single_word_city_allow,
            'dict_street_like_terms': dict_street_like_terms,
            'dict_anchor_tokens': dict_anchor_tokens,
            'dict_require_anchor_for_city_inference': dict_require_anchor_for_city_inference,
            'dict_reject_if_only_street_like': dict_reject_if_only_street_like,
        }

    cache = match_address._cache
    dim_work = cache['dim_work']
    dim_city_variant_map = cache['dim_city_variant_map']
    dim_city_raw_map = cache['dim_city_raw_map']
    barangay_set = cache['barangay_set']
    brgy_token_max = cache['brgy_token_max']
    barangay_to_cityset = cache['barangay_to_cityset']
    dict_city_alias_map = cache['dict_city_alias_map']
    dict_city_display_map = cache['dict_city_display_map']
    dict_city_province_map = cache['dict_city_province_map']
    dict_city_region_map = cache['dict_city_region_map']
    dict_city_terms = cache['dict_city_terms']
    dict_province_alias_map = cache['dict_province_alias_map']
    dict_province_terms = cache['dict_province_terms']
    dict_area_term_to_entries = cache['dict_area_term_to_entries']
    dict_area_terms = cache['dict_area_terms']
    dict_single_word_city_allow = cache['dict_single_word_city_allow']
    dict_street_like_terms = cache['dict_street_like_terms']
    dict_anchor_tokens = cache['dict_anchor_tokens']
    dict_require_anchor_for_city_inference = cache['dict_require_anchor_for_city_inference']
    dict_reject_if_only_street_like = cache['dict_reject_if_only_street_like']

    lookup_text = _canon(address_norm)
    anchor_pattern = r'\b(' + '|'.join(re.escape(tok) for tok in sorted(dict_anchor_tokens, key=len, reverse=True)) + r')\b' if dict_anchor_tokens else None
    has_city_keyword = bool(re.search(r'\b(CITY|MUNICIPALITY|MUN)\b', lookup_text))
    has_brgy_keyword = bool(re.search(r'\b(BRGY|BARANGAY|BGY)\b', lookup_text))
    has_province_keyword = bool(re.search(r'\b(PROVINCE|PROV)\b', lookup_text))
    has_anchor_token = bool(anchor_pattern and re.search(anchor_pattern, lookup_text))
    has_geo_anchor = has_city_keyword or has_brgy_keyword or has_province_keyword or has_anchor_token

    def _find_explicit_city(text_canon: str) -> tuple[Optional[str], Optional[str]]:
        padded = f' {text_canon} '
        for term in dict_city_terms:
            if f' {term} ' not in padded:
                continue
            if len(term.split()) == 1 and term not in dict_single_word_city_allow:
                continue
            return dict_city_alias_map[term], term
        return None, None

    explicit_city_can, explicit_city_term = _find_explicit_city(lookup_text)

    def _find_explicit_province(text_canon: str, city_hint: bool) -> tuple[Optional[str], Optional[str]]:
        padded = f' {text_canon} '
        for term in dict_province_terms:
            if f' {term} ' not in padded:
                continue
            if has_province_keyword or city_hint or has_anchor_token:
                return dict_province_alias_map[term], term
        return None, None

    def _find_explicit_area_city(text_canon: str):
        if dict_require_anchor_for_city_inference and not has_geo_anchor:
            return None, None, False
        padded = f' {text_canon} '
        for term in sorted(dict_area_terms, key=len, reverse=True):
            if f' {term} ' not in padded:
                continue
            entries = dict_area_term_to_entries.get(term, [])
            city_candidates = []
            for e in entries:
                if not e.get('infer_allowed', False):
                    continue
                if e.get('requires_context', False) and not has_geo_anchor:
                    continue
                city_candidates.append(e['city_can'])
            uniq = sorted(set(city_candidates))
            if len(uniq) == 1:
                return uniq[0], term, False
            if len(uniq) > 1:
                return None, term, True
        return None, None, False

    area_city_can, area_term, area_ambiguous = _find_explicit_area_city(lookup_text)
    explicit_prov_can, explicit_prov_term = _find_explicit_province(lookup_text, bool(explicit_city_can or area_city_can))

    if (
        dict_reject_if_only_street_like
        and not explicit_city_can
        and not area_city_can
        and not area_ambiguous
        and not has_geo_anchor
        and any(f' {t} ' in f' {lookup_text} ' for t in dict_street_like_terms)
    ):
        result.update(
            city_required_pass=False,
            city_signal=None,
            invalid_reason='street_only_no_geo_anchor',
            confidence='low',
            needs_api=False,
        )
        return result

    if area_ambiguous:
        result.update(
            city_required_pass=False,
            city_signal=None,
            invalid_reason='ambiguous_area_multi_city',
            confidence='low',
            needs_api=False,
        )
        return result

    city_signal_can = explicit_city_can or area_city_can
    city_signal_source = 'explicit_city' if explicit_city_can else ('dict_area' if area_city_can else None)
    city_signal_term = explicit_city_term if explicit_city_can else area_term

    if not city_signal_can:
        result.update(
            city_required_pass=False,
            city_signal=None,
            invalid_reason='missing_explicit_city',
            confidence='low',
            needs_api=False,
        )
        return result

    dict_city_prov_can = dict_city_province_map.get(city_signal_can)
    dict_city_region_can = dict_city_region_map.get(city_signal_can)

    if explicit_prov_can and dict_city_prov_can and explicit_prov_can != dict_city_prov_can:
        result.update(
            city_required_pass=False,
            city_signal=dict_city_display_map.get(city_signal_can, city_signal_can),
            invalid_reason='province_city_conflict',
            confidence='low',
            needs_api=False,
        )
        return result

    selected_prov_can = explicit_prov_can or dict_city_prov_can

    city_lookup_can = dim_city_variant_map.get(city_signal_can)
    if city_lookup_can is None and city_signal_can in set(dim_work['_city_canon'].unique().tolist()):
        city_lookup_can = city_signal_can
    if city_lookup_can is None:
        result.update(
            city_required_pass=False,
            city_signal=dict_city_display_map.get(city_signal_can, city_signal_can),
            invalid_reason='city_not_resolved',
            confidence='low',
            needs_api=False,
        )
        return result

    city_subset = dim_work[dim_work['_city_canon'] == city_lookup_can].copy()
    if city_subset.empty:
        result.update(
            city_required_pass=False,
            city_signal=dict_city_display_map.get(city_signal_can, city_signal_can),
            invalid_reason='city_not_resolved',
            confidence='low',
            needs_api=False,
        )
        return result

    if selected_prov_can:
        exact_subset = city_subset[city_subset['_prov_canon'] == selected_prov_can].copy()
        if not exact_subset.empty:
            city_subset = exact_subset

    chosen = city_subset.iloc[0]
    city_display = dict_city_display_map.get(city_signal_can, chosen['city_name'])
    province_display = (
        dict_city_prov_can
        or explicit_prov_can
        or chosen['province_name']
    )
    region_display = dict_city_region_can or chosen['region_name']

    explicit_brgy_name = None
    explicit_brgy_code = None
    if city_signal_source == 'dict_area' and city_signal_term:
        explicit_brgy_name = city_signal_term

    result.update(
        city_name=city_display,
        city_code=int(chosen['city_code']),
        city_score=100 if city_signal_source == 'explicit_city' else 95,
        province_name=province_display,
        province_code=int(chosen['province_code']),
        province_score=100 if explicit_prov_can else (90 if selected_prov_can else 0),
        region_name=region_display,
        region_code=int(chosen['region_code']),
        barangay_name=explicit_brgy_name,
        barangay_code=explicit_brgy_code,
        psgc_10_digit=int(chosen['psgc_10_digit']),
        confidence='high' if city_signal_source == 'explicit_city' else 'medium',
        needs_api=False,
        city_required_pass=True,
        city_signal=city_display,
        invalid_reason=None,
    )
    return result


# -- Generalized strict-evidence regression tests -------------------------------
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
    'LIONS PARK RES SUN VALLEY',
    'House 4 Adolfo Compound, Highway 11 St Brgy Talamban, City of Cebu',
    'BGC 30TH STREET',
    'TONDO MANILA',
    'BINONDO MANILA',
    '2248 Angel Linao St',
    'AURORA BLVD COR 20TH AVE',
    'NEW MANILA near E. Rodriguez',
    'BRGY SUN VALLEY',
    'BRGY STA CRUZ',
    '123 5TH FLOOR ABC BUILDING',
    'CUBAO',
]

for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  City signal: {ans["city_signal"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print(f'  Invalid rsn: {ans["invalid_reason"]}')
    print('-' * 80)


Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  City signal: QUEZON CITY
  City       : None
  Province   : None
  Barangay   : None
  Confidence : low
  Invalid rsn: province_city_conflict
--------------------------------------------------------------------------------
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SIGNAL VILLAGE TAGUIG CITY
  City signal: CITY OF TAGUIG
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Barangay   : None
  Confidence : high
  Invalid rsn: None
--------------------------------------------------------------------------------
Test 3
  Input      : LIONS PARK RES SUN VALLEY
  Normalized : LIONS PARK RES SUN VALLEY
  City signal: None
  City       : None
  Province   : None
  Barangay   : None
  Confidence : low
  Invalid rsn: missing_explicit_city
--------------------------

In [350]:
if '_match_address_base' not in globals():
    _match_address_base = match_address


def _canon_local(text: str) -> str:
    import unicodedata
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def match_address(address_norm: str, dim: pd.DataFrame, cities: list, provinces: list, regions: list, barangays: list):
    lookup_text = _canon_local(address_norm)
    padded_lookup = f' {lookup_text} '

    has_city_keyword = bool(re.search(r'\b(CITY|MUNICIPALITY|MUN)\b', lookup_text))
    has_brgy_keyword = bool(re.search(r'\b(BRGY|BARANGAY|BGY)\b', lookup_text))
    has_province_keyword = bool(re.search(r'\b(PROVINCE|PROV)\b', lookup_text))
    has_geo_anchor = has_city_keyword or has_brgy_keyword or has_province_keyword

    street_like_terms = {
        'ST', 'STREET', 'AVE', 'AVENUE', 'ROAD', 'RD', 'BLK', 'BLOCK', 'LOT',
        'UNIT', 'PHASE', 'SUBD', 'SUBDIVISION', 'RESIDENCE', 'RESIDENCES',
        'VILLAGE', 'VILLAGES', 'COMPOUND', 'BUILDING', 'BLDG', 'APARTMENT',
        'APARTMENTS', 'HOMES', 'HOME', 'TOWER', 'PARK', 'ESTATE', 'PUROK',
        'ZONE', 'COR', 'CORNER', 'HIGHWAY', 'HWY', 'LANE', 'LN', 'BARANGAY',
        'BRGY', 'BGY', 'POBLACION', 'PUEBLO', 'CAPITOL', 'CAPITOLIO',
    }
    has_street_like = any(f' {term} ' in padded_lookup for term in street_like_terms)

    if not has_geo_anchor and has_street_like:
        return {
            'city_name': None,
            'city_code': None,
            'city_score': 0,
            'province_name': None,
            'province_code': None,
            'province_score': 0,
            'region_name': None,
            'region_code': None,
            'barangay_name': None,
            'barangay_code': None,
            'psgc_10_digit': None,
            'confidence': 'low',
            'needs_api': False,
            'city_required_pass': False,
            'city_signal': None,
            'invalid_reason': 'street_only_no_geo_anchor',
        }

    result = _match_address_base(address_norm, dim, cities, provinces, regions, barangays)

    if result.get('invalid_reason') == 'province_city_conflict' and result.get('city_signal'):
        city_can = _canon_local(result['city_signal'])
        city_prov_can = dict_city_province_map.get(city_can)
        city_region_can = dict_city_region_map.get(city_can)
        city_display = dict_city_display_map.get(city_can)
        embedded_prov_terms = [
            term for term in dict_province_terms
            if term != city_prov_can and f' {term} ' in f' {city_can} '
        ]
        if city_can and city_display and city_prov_can and embedded_prov_terms:
            city_rows = dim.copy()
            city_rows['_city_canon'] = city_rows['city_name'].fillna('').astype(str).apply(_canon_local)
            city_rows['_prov_canon'] = city_rows['province_name'].fillna('').astype(str).apply(_canon_local)
            city_rows = city_rows[city_rows['_city_canon'] == city_can].copy()
            if not city_rows.empty:
                exact_rows = city_rows[city_rows['_prov_canon'] == city_prov_can].copy()
                chosen = exact_rows.iloc[0] if not exact_rows.empty else city_rows.iloc[0]
                recovered = dict(result)
                recovered['city_name'] = chosen['city_name']
                recovered['city_code'] = int(chosen['city_code'])
                recovered['province_name'] = city_prov_can or chosen['province_name']
                recovered['province_code'] = int(chosen['province_code'])
                recovered['region_name'] = city_region_can or chosen['region_name']
                recovered['region_code'] = int(chosen['region_code'])
                recovered['city_required_pass'] = True
                recovered['invalid_reason'] = None
                recovered['confidence'] = 'high'
                recovered['city_signal'] = city_display
                return recovered
    return result


In [351]:
# Quick validation after the province-overlap recovery wrapper.
for raw in [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    'Cainta Rizal',
    '112 Ballecer st Zone South Signal Village Taguig',
    'BGC 30TH STREET',
]:
    norm = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(norm, dim, cities, provinces, regions, barangays)
    print(raw)
    print('  norm :', norm)
    print('  city :', ans['city_name'])
    print('  prov :', ans['province_name'])
    print('  conf :', ans['confidence'])
    print('  reas :', ans['invalid_reason'])
    print('-' * 60)


19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  norm : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  city : QUEZON CITY
  prov : METRO MANILA
  conf : high
  reas : None
------------------------------------------------------------
Cainta Rizal
  norm : CAINTA CITY RIZAL
  city : CAINTA
  prov : RIZAL
  conf : high
  reas : None
------------------------------------------------------------
112 Ballecer st Zone South Signal Village Taguig
  norm : 112 BALLECER STREET ZONE SIGNAL VILLAGE TAGUIG CITY
  city : CITY OF TAGUIG
  prov : METRO MANILA
  conf : high
  reas : None
------------------------------------------------------------
BGC 30TH STREET
  norm : BGC 30TH STREET
  city : None
  prov : None
  conf : low
  reas : street_only_no_geo_anchor
------------------------------------------------------------


In [352]:
def _join_query_parts(parts):
    cleaned = [str(part).strip() for part in parts if part is not None and str(part).strip()]
    return re.sub(r'\s+', ' ', ' '.join(cleaned)).strip()


def _nominatim_lookup(query_text: str) -> dict | None:
    params = {
        'q': query_text,
        'format': 'jsonv2',
        'addressdetails': 1,
        'limit': 1,
        'countrycodes': 'ph',
    }
    if NOMINATIM_EMAIL:
        params['email'] = NOMINATIM_EMAIL
    url = NOMINATIM_URL + '?' + urllib.parse.urlencode(params)
    request = urllib.request.Request(
        url,
        headers={
            'User-Agent': NOMINATIM_USER_AGENT,
            'Accept': 'application/json',
        },
    )
    with urllib.request.urlopen(request, timeout=NOMINATIM_TIMEOUT) as response:
        payload = json.loads(response.read().decode('utf-8'))
    return payload[0] if payload else None


def _extract_nominatim_parts(payload: dict) -> dict:
    address = payload.get('address', {}) if isinstance(payload, dict) else {}
    city = (
        address.get('city')
        or address.get('town')
        or address.get('municipality')
        or address.get('village')
        or address.get('suburb')
        or address.get('city_district')
        or address.get('borough')
    )
    province = address.get('state') or address.get('province') or address.get('state_district')
    region = address.get('region') or address.get('ISO3166-2-lvl4')
    return {
        'api_city': city,
        'api_province': province,
        'api_region': region,
        'api_display_name': payload.get('display_name'),
        'api_lat': payload.get('lat'),
        'api_lon': payload.get('lon'),
    }


def resolve_with_osm_nominatim(
    record: dict,
    dim: pd.DataFrame,
    cities: list,
    provinces: list,
    regions: list,
    barangays: list,
    compiled_re,
    replace_map: dict,
    api_query_mode: str = 'freeform',
) -> dict:
    result = {
        'api_status': 'skipped',
        'api_query': None,
        'api_error': None,
        'api_match_address': None,
        'api_city': None,
        'api_province': None,
        'api_region': None,
        'osm_display': None,
        'osm_lat': None,
        'osm_lon': None,
    }

    needs_api = bool(record.get('needs_api')) or record.get('confidence') not in {'high'}
    if not USE_API or not needs_api:
        return result

    if api_query_mode == 'structured':
        query_text = _join_query_parts([
            record.get('address_normalized'),
            record.get('city_name'),
            record.get('province_name'),
            record.get('region_name'),
            'Philippines',
        ])
    else:
        query_text = _join_query_parts([
            record.get('address_normalized') or record.get('original_address'),
            'Philippines',
        ])

    result['api_query'] = query_text
    if not query_text:
        result['api_status'] = 'empty_query'
        return result

    try:
        payload = _nominatim_lookup(query_text)
    except Exception as exc:
        result['api_status'] = 'error'
        result['api_error'] = str(exc)
        return result

    if not payload:
        result['api_status'] = 'not_found'
        return result

    result.update(_extract_nominatim_parts(payload))
    api_match_address = _join_query_parts([
        result['api_city'],
        result['api_province'],
        result['api_region'],
        result['osm_display'],
    ])
    result['api_match_address'] = api_match_address

    if not api_match_address:
        result['api_status'] = 'not_found'
        return result

    api_norm = normalize_alias(api_match_address, compiled_re, replace_map)
    api_mapped = match_address(api_norm, dim, cities, provinces, regions, barangays)
    if api_mapped.get('city_required_pass'):
        result.update({
            'api_status': 'mapped',
            'city_name': api_mapped.get('city_name'),
            'city_code': api_mapped.get('city_code'),
            'province_name': api_mapped.get('province_name'),
            'province_code': api_mapped.get('province_code'),
            'region_name': api_mapped.get('region_name'),
            'region_code': api_mapped.get('region_code'),
            'barangay_name': api_mapped.get('barangay_name'),
            'barangay_code': api_mapped.get('barangay_code'),
            'psgc_10_digit': api_mapped.get('psgc_10_digit'),
            'city_required_pass': True,
            'confidence': 'api',
            'invalid_reason': None,
            'city_signal': api_mapped.get('city_signal'),
        })
    else:
        result['api_status'] = 'found_unmapped'

    return result


## Cell 7 — Run the Full Pipeline

Read raw rows, normalize and complete the address, enrich with OSM Nominatim when needed, then map the API result into `dim_location` codes for export.

In [353]:
import time

t_start = time.time()

if 'area_dict_ctx' not in globals() or not isinstance(globals().get('area_dict_ctx'), dict):
    area_dict_ctx = load_area_dictionary(AREA_DICT_FILE)
    if isinstance(area_dict_ctx, dict):
        globals().update(area_dict_ctx)

if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    log.info(f'Loading batch files: {len(existing_paths)} found')
    frames = []
    for p in existing_paths:
        tmp = pd.read_excel(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df_input = pd.concat(frames, ignore_index=True)
else:
    log.info(f'Loading {INPUT_FILE} ...')
    df_input = pd.read_excel(INPUT_FILE)

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# Stage 1: raw load and normalization using alias + dictionary context.
addresses = df_input[addr_col].fillna('').astype(str).tolist()
normalized = [normalize_alias(x, compiled_re, replace_map) for x in addresses]
df_input['address_normalized'] = normalized

records = []
api_hits = 0

for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Pipeline', unit='row')):
    record = {
        'original_address': addresses[i],
        'address_normalized': addr_norm,
        'api_status': 'skipped',
        'api_query': None,
        'api_error': None,
        'api_match_address': None,
        'api_city': None,
        'api_province': None,
        'api_region': None,
        'osm_display': None,
        'osm_lat': None,
        'osm_lon': None,
    }
    if '_batch_file' in df_input.columns:
        record['batch_file'] = df_input.iloc[i]['_batch_file']

    # Stage 2: complete abbreviated, acronym, or aliased place names before API lookup.
    record.update(match_address(
        addr_norm,
        dim, cities, provinces, regions, barangays,
    ))
    record['needs_api'] = USE_API and record.get('confidence') != 'high'

    # Stage 3: OSM Nominatim lookup for unresolved rows.
    api_result = resolve_with_osm_nominatim(
        record,
        dim,
        cities,
        provinces,
        regions,
        barangays,
        compiled_re,
        replace_map,
        api_query_mode=API_QUERY_MODE,
    )
    if api_result.get('api_status') == 'mapped':
        api_hits += 1
    record.update(api_result)

    # Stage 4: retain API-mapped PSGC values when available.
    record['is_valid'] = bool(record.get('city_required_pass')) and bool(record.get('city_name')) and record.get('city_code') is not None
    records.append(record)

    if record.get('needs_api') and NOMINATIM_PAUSE_SECONDS:
        time.sleep(NOMINATIM_PAUSE_SECONDS)

high_conf = [r for r in records if r.get('is_valid')]
low_conf = [r for r in records if not r.get('is_valid')]

out_df = pd.DataFrame(records)

if 'city_required_pass' not in out_df.columns:
    out_df['city_required_pass'] = False
out_df['city_required_pass'] = out_df['city_required_pass'].fillna(False).astype(bool)
out_df['is_valid'] = out_df['is_valid'].fillna(False).astype(bool)

t_fuzzy = time.time() - t_start
log.info(f'Pipeline completed in {t_fuzzy:.1f}s')
log.info(f'  Verified / mapped : {len(high_conf):,}')
log.info(f'  Unmapped          : {len(low_conf):,}')
log.info(f'  API hits          : {api_hits:,}')

# -- Preview ------------------------------------------------------------------
preview_cols = [
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'api_status', 'city_score'
]
if records and 'batch_file' in records[0]:
    preview_cols = ['batch_file'] + preview_cols

df_preview = out_df[preview_cols]
display(df_preview.head(20))

11:26:14  INFO      Loading ..\..\data\sample\copy_ps_address_v3.xlsx ...
11:26:17  INFO      Limiting to 10000 rows (MAX_ROWS is set)
11:26:17  INFO      Address column: "original_address"  |  Total rows: 10,000


Pipeline:   0%|          | 0/10000 [00:00<?, ?row/s]

16:23:31  INFO      Pipeline completed in 17837.3s
16:23:31  INFO        Verified / mapped : 4,318
16:23:31  INFO        Unmapped          : 5,682
16:23:31  INFO        API hits          : 1,930


,original_address,address_normalized,city_name,province_name,confidence,api_status,city_score
0,Tagumpay sindalan CSFP,TAGUMPAY SINDALAN SAN FERNANDO CITY,CITY OF SAN FERNANDO,LA UNION,high,skipped,100
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",BLOCK 16 LOT 4 QUIOTAN STREET PHIL AM LIFE VIL...,NaN,NaN,low,not_found,0
2,Cainta Rizal,CAINTA CITY RIZAL,CAINTA,RIZAL,high,skipped,100
3,2079 a elias st sta cruz,2079 A ELIAS STREET SANTA CRUZ,NaN,NaN,low,found_unmapped,0
4,Swani Hardware 77 tirso neri street CDOC,SWANI HARDWARE 77 TIRSO NERI STREET CAGAYAN DE...,CITY OF CAGAYAN DE ORO,MISAMIS ORIENTAL,high,skipped,0
5,ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalai...,ADP BUILDING GOV A LUGAY AVENUE MANGGA DOS MAT...,TUGUEGARAO CITY,CAGAYAN,high,skipped,100
6,tapaz capiz,TAPAZ CAPIZ,TAPAZ,CAPIZ,high,skipped,100
7,4662PagAsa St V mapa Sta Mesa Manila,4662PAGASA STREET V MAPA Santa Mesa MANILA,NaN,NaN,low,not_found,0
8,CENTRO ENRILE CAGAYAN,CENTRO ENRILE CAGAYAN,ENRILE,CAGAYAN,high,skipped,100
9,tayuman manila,TAYUMAN MANILA,NaN,NaN,low,found_unmapped,0


## Cell 8 — Merge and Finalize Results

Combine the raw, normalized, and API-enriched rows into a single output table and mark rows as verified when PSGC mapping is available.

In [354]:
all_records = high_conf + low_conf
out_df = pd.DataFrame(all_records)

if 'city_required_pass' not in out_df.columns:
    out_df['city_required_pass'] = False
out_df['city_required_pass'] = out_df['city_required_pass'].fillna(False).astype(bool)

# Final validity flag with strict city-evidence requirement
out_df['is_valid'] = out_df['city_required_pass']

# Confidence value counts
print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'invalid_reason' in out_df.columns:
    print('\nInvalid reason distribution:')
    print(out_df['invalid_reason'].fillna('resolved').value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


Confidence distribution:
confidence
low       5682
high      1948
api       1930
medium     440

Invalid reason distribution:
invalid_reason
resolved                     4318
street_only_no_geo_anchor    3992
missing_explicit_city        1488
province_city_conflict        100
city_not_resolved              98
ambiguous_area_multi_city       4

Valid   : 4,318
Invalid : 5,682
Total   : 10,000


In [355]:
# City-only sanity check: barangay should remain blank when not explicit
city_only_mask = out_df['original_address'].astype(str).str.lower().isin(['vigan ilocos sur', 'loilo city'])
display(out_df.loc[city_only_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code


In [356]:
# Detailed-address check: barangay should still be populated when appropriate
detail_mask = out_df['original_address'].astype(str).str.contains(
    'Batasan|South Signal|Kim View|sapang palay', case=False, na=False
)
display(out_df.loc[detail_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code
520,23 Aguilar Street Zone 4 Brgy South Signal Tag...,AGUILAR,PANGASINAN,NaN,None
1749,sapang palay sjdm,CITY OF SAN JOSE DEL MONTE,BULACAN,NaN,None
2748,FRIENDSHIP ST ZONE 6 SOUTH SIGNAL,CITY OF TAGUIG,METRO MANILA,NaN,None
3069,Sapang palay proper,BULACAN,BULACAN,NaN,None
3414,Rantio Sapang Palay Bulacan,BULACAN,BULACAN,NaN,None
4143,#62 sto.nino street south signal taguig city,SANTO NIÑO,CAGAYAN,NaN,None
4968,1 Aguinaldo st South Signal,NaN,NaN,NaN,None
6747,Filinvest Homes 1 1 important cuz subd aft...,NaN,NaN,NaN,None
7574,Unit 7 joanna executive home batasan,NaN,NaN,NaN,None
7997,197 Lot B Sapang palay Proper,NaN,NaN,NaN,None


In [357]:
# Before-vs-after change report (fuzzy baseline vs API-final)

# Fallback for fuzzy-only mode (when pre_api_df was never created)
pre_api_obj = globals().get('pre_api_df', None)
if isinstance(pre_api_obj, pd.DataFrame) and not pre_api_obj.empty:
    baseline_df = pre_api_obj.copy()
else:
    baseline_df = out_df.copy()
    print('pre_api_df not found; using out_df as baseline (fuzzy-only fallback).')

compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']

# Ensure required columns exist in both frames
for c in compare_cols:
    if c not in baseline_df.columns:
        baseline_df[c] = 'skipped' if c == 'api_status' else None
    if c not in out_df.columns:
        out_df[c] = 'skipped' if c == 'api_status' else None

base = baseline_df[['original_address'] + compare_cols].copy()
final = out_df[['original_address'] + compare_cols].copy()

# Prevent many-to-many merge explosion on duplicate addresses
base['_row_id'] = base.groupby('original_address').cumcount()
final['_row_id'] = final.groupby('original_address').cumcount()

base = base.rename(columns={
    'city_name': 'city_before',
    'province_name': 'province_before',
    'confidence': 'confidence_before',
    'api_status': 'api_status_before',
})

final = final.rename(columns={
    'city_name': 'city_after',
    'province_name': 'province_after',
    'confidence': 'confidence_after',
    'api_status': 'api_status_after',
})

change_df = base.merge(final, on=['original_address', '_row_id'], how='inner').drop(columns=['_row_id'])

changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)

changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)

pre_api_df not found; using out_df as baseline (fuzzy-only fallback).
Rows changed after API verification: 0 / 10,000


,original_address,city_before,province_before,confidence_before,api_status_before,city_after,province_after,confidence_after,api_status_after


In [358]:
# Compact delta view
compact = changed_rows[[
    'original_address',
    'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

# Highlight the user's example row
mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address',
        'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])

,original_address,city_before,city_after,province_before,province_after,api_status_before,api_status_after


## Cell 9 — Export to Excel

Output three files:
1. `verified_addresses_parts_XXX_YYY.xlsx` (green tab) — Valid addresses only
2. `invalid_addresses_parts_XXX_YYY.xlsx` (red tab) — Invalid addresses  
3. `combined_addresses_parts_XXX_YYY.xlsx` (blue tab) — All addresses with diagnostics

Base columns: Original Address, Barangay Code/Name, City Code/Name, Province Code/Name, Region. Combined file adds: Normalized Address, PSGC, Confidence, City/Province Scores, Status.

In [359]:
from pathlib import Path

# -- Output targets ------------------------------------------------------------
out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

# Build suffix from detected input parts (e.g. parts_001_010)
batch_nums = []
for p in input_paths:
    m = re.search(r'part_(\d+)', p.name.lower())
    if m:
        batch_nums.append(int(m.group(1)))

if batch_nums:
    batch_suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    batch_suffix = f'batch_{len(input_paths):03d}'
else:
    batch_suffix = 'single'

VERIFIED_FILE = str(out_verified_dir / f'verified_addresses_{batch_suffix}.xlsx')
INVALID_FILE = str(out_invalid_dir / f'invalid_addresses_{batch_suffix}.xlsx')
COMBINED_FILE = str(out_root / f'combined_addresses_{batch_suffix}.xlsx')

# -- Column order required by business template (image) -----------------------
BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

# Build standardized base table from pipeline output
base_df = out_df.copy()
base_df = base_df.rename(columns={
    'original_address': 'Original_Address',
    'barangay_name': 'barangay',
    'city_name': 'city',
    'province_name': 'province',
})

# Ensure expected columns exist even when null in invalid rows
for c in BASE_EXPORT_COLS:
    if c not in base_df.columns:
        base_df[c] = None

verified_df = base_df[base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df = base_df[~base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

# Combined file keeps additional diagnostics + API status
combined_cols = [
    'Original_Address',
    'Normalized_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
    'psgc_10',
    'confidence',
    'city_score',
    'province_score',
    'api_status',
    'api_query',
    'api_match_address',
    'api_city',
    'api_province',
    'api_region',
    'osm_display',
    'osm_lat',
    'osm_lon',
    'api_error',
    'status',
]

combined_df = base_df.copy()
combined_df['Normalized_Address'] = combined_df.get('address_normalized')
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status'] = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows -> {path}')


write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df, INVALID_FILE, 'Invalid', '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  ✅  {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  📄  {COMBINED_FILE}  ({len(combined_df):,} rows)')

16:23:34  INFO      Wrote 4,318 rows -> ..\..\data\output\verified\verified_addresses_single.xlsx
16:23:35  INFO      Wrote 5,682 rows -> ..\..\data\output\invalid\invalid_addresses_single.xlsx
16:23:39  INFO      Wrote 10,000 rows -> ..\..\data\output\combined_addresses_single.xlsx



Total elapsed: 17845.8s  (297.43 min)
Output files:
  ✅  ..\..\data\output\verified\verified_addresses_single.xlsx  (4,318 rows)
  ⚠️   ..\..\data\output\invalid\invalid_addresses_single.xlsx   (5,682 rows)
  📄  ..\..\data\output\combined_addresses_single.xlsx  (10,000 rows)


## Cell 10 — Results Summary

Display summary: total rows, verified/invalid counts, confidence distribution, average city score for high-confidence rows. Preview first 10 verified and all invalid rows.

In [360]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<20} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf): {high_c["city_score"].mean():.1f}')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


════════════════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Input rows        :   10,000
  Verified          :    4,318  (43.2%)
  Invalid           :    5,682  (56.8%)

  Confidence breakdown:
    low                   5,682  (56.8%)
    high                  1,948  (19.5%)
    api                   1,930  (19.3%)
    medium                  440  (4.4%)

  Avg city score (high-conf): 88.6
  Total elapsed     : 17845.8s
════════════════════════════════════════════════════════════════════════════════

── Verified (first 10 rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,Tagumpay sindalan CSFP,None,NaN,14.0,CITY OF SAN FERNANDO,33.0,LA UNION,1.0,REGION I ILOCOS REGION
1,Cainta Rizal,None,NaN,5.0,CAINTA,58.0,RIZAL,4.0,REGION IV-A (CALABARZON)
2,Swani Hardware 77 tirso neri street CDOC,None,NaN,0.0,CITY OF CAGAYAN DE ORO,43.0,MISAMIS ORIENTAL,10.0,REGION X NORTHERN MINDANAO
3,ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalai...,None,NaN,29.0,TUGUEGARAO CITY,15.0,CAGAYAN,2.0,REGION II (CAGAYAN VALLEY)
4,tapaz capiz,None,NaN,17.0,TAPAZ,19.0,CAPIZ,6.0,REGION VI (WESTERN VISAYAS)
5,CENTRO ENRILE CAGAYAN,None,NaN,12.0,ENRILE,15.0,CAGAYAN,2.0,REGION II (CAGAYAN VALLEY)
6,Barrio Calinog Iloilo,None,NaN,22.0,CITY OF ILOILO,30.0,ILOILO,6.0,REGION VI WESTERN VISAYAS
7,ANA ROS VILLAGE MANDURRIAO,None,NaN,22.0,CITY OF ILOILO,30.0,ILOILO,6.0,REGION VI WESTERN VISAYAS
8,CONCEPCION BALIUAG,None,NaN,15.0,CONCEPCION,30.0,ILOILO,6.0,REGION VI (WESTERN VISAYAS)
9,"KOREAN CRAVE UNITOP BALIUAG, BULACAN",None,NaN,5.0,BULACAN,14.0,BULACAN,3.0,REGION III (CENTRAL LUZON)



── Invalid (all rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2079 a elias st sta cruz,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4662PagAsa St V mapa Sta Mesa Manila,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,tayuman manila,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CAGAYAN 3500 GOMEZ STREET,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
5677,BULO SUKAN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5678,villa antonio Subd,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5679,BLK 8 LOT 5 PHASE 3 BEVERLY HOMES SUBD. PRENZA...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5680,Prob Park Dilg,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cell 11 — Reusable Pipeline Function

Define `run_pipeline()` for quick re-runs after config changes. The function follows the staged flow: raw load, normalization, address completion, OSM Nominatim lookup, and `dim_location` mapping.

In [361]:
def run_pipeline(
    input_file   = INPUT_FILE,
    dim_path     = DIM_LOCATION,
    alias_path   = ALIAS_FILE,
    out_verified = OUT_VERIFIED,
    out_invalid  = OUT_INVALID,
    max_rows     = MAX_ROWS,
    use_api      = USE_API,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()

    area_ctx = load_area_dictionary(AREA_DICT_FILE)
    if isinstance(area_ctx, dict):
        globals().update(area_ctx)

    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(
        dim_path, alias_path
    )
    _re, _rmap = build_alias_regex(_alias)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]

    addresses = df[col].fillna('').astype(str).tolist()
    normalized = [normalize_alias(x, _re, _rmap) for x in addresses]

    recs = []
    for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Pipeline', unit='row')):
        rec = {
            'original_address': addresses[i],
            'address_normalized': addr_norm,
            'api_status': 'skipped',
            'api_query': None,
            'api_error': None,
            'api_match_address': None,
            'api_city': None,
            'api_province': None,
            'api_region': None,
            'osm_display': None,
            'osm_lat': None,
            'osm_lon': None,
        }
        rec.update(match_address(
            addr_norm,
            _dim, _cities, _provinces, _regions, _barangays,
        ))
        rec['needs_api'] = use_api and rec.get('confidence') != 'high'

        api_result = resolve_with_osm_nominatim(
            rec,
            _dim,
            _cities,
            _provinces,
            _regions,
            _barangays,
            _re,
            _rmap,
            api_query_mode=api_query_mode,
        )
        rec.update(api_result)
        rec['is_valid'] = bool(rec.get('city_required_pass')) and bool(rec.get('city_name')) and rec.get('city_code') is not None
        recs.append(rec)

        if rec.get('needs_api') and NOMINATIM_PAUSE_SECONDS:
            time.sleep(NOMINATIM_PAUSE_SECONDS)

    merged = pd.DataFrame(recs)
    if 'city_required_pass' not in merged.columns:
        merged['city_required_pass'] = False
    merged['city_required_pass'] = merged['city_required_pass'].fillna(False).astype(bool)
    merged['is_valid'] = merged['is_valid'].fillna(False).astype(bool)

    base_cols = [
        'original_address',
        'address_normalized',
        'barangay_code', 'barangay_name',
        'city_code', 'city_name',
        'province_code', 'province_name',
        'region_code', 'region_name',
        'psgc_10_digit',
        'confidence',
        'api_status',
        'osm_display',
        'osm_lat',
        'osm_lon',
        'invalid_reason',
    ]
    base_cols = [c for c in base_cols if c in merged.columns]

    v_df = merged[merged['is_valid']][base_cols].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][base_cols].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid, 'Invalid', '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df, merged


# Uncomment to run:
# verified_df, invalid_df, out_df = run_pipeline()
# verified_df, invalid_df, out_df = run_pipeline(max_rows=1000)


In [362]:
# Cell 13 -- Invalid pattern diagnostics (batch profile)
diag_df = out_df.copy()
diag_df['orig'] = diag_df['original_address'].fillna('').astype(str)
diag_df['norm'] = diag_df['address_normalized'].fillna('').astype(str)
diag_df['invalid_reason'] = diag_df.get('invalid_reason', pd.Series([None] * len(diag_df))).fillna('resolved')

invalid_only = diag_df[~diag_df['is_valid']].copy()
print(f'Total rows: {len(diag_df):,}')
print(f'Invalid rows: {len(invalid_only):,}')

print('\nTop invalid reasons:')
print(invalid_only['invalid_reason'].value_counts().head(10).to_string())

# Lightweight pattern flags for error clustering
pat = pd.DataFrame(index=invalid_only.index)
pat['has_brgy_word'] = invalid_only['norm'].str.contains(r'\b(?:BRGY|BARANGAY|BGY)\b', regex=True, na=False)
pat['has_city_word'] = invalid_only['norm'].str.contains(r'\b(?:CITY|MUNICIPALITY|MUN)\b', regex=True, na=False)
pat['has_numbers'] = invalid_only['norm'].str.contains(r'\d', regex=True, na=False)
pat['short_addr'] = invalid_only['norm'].str.len() <= 28
pat['street_only_tokens'] = invalid_only['norm'].str.contains(
    r'\b(?:ST|STREET|AVE|AVENUE|ROAD|RD|BLK|LOT|UNIT|PHASE|B\d+|L\d+)\b', regex=True, na=False
)

cluster = pd.DataFrame({
    'missing_city_word': (~pat['has_city_word']).astype(int),
    'with_brgy_word': pat['has_brgy_word'].astype(int),
    'street_only_tokens': pat['street_only_tokens'].astype(int),
    'short_addr': pat['short_addr'].astype(int),
})
print('\nPattern prevalence among invalid rows:')
print(cluster.mean().mul(100).round(1).astype(str) + '%')

print('\nSample invalid addresses (first 30):')
display(invalid_only[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

# Show likely false-invalid candidates (have city-like cues but still invalid)
candidates = invalid_only[
    invalid_only['norm'].str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|CEBU|QC|QUEZON|MANILA|MAKATI|TAGUIG)\b',
        regex=True, na=False
    )
]
print(f'\nPotentially recoverable invalids with city-like cues: {len(candidates):,}')
display(candidates[['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']].head(30))

Total rows: 10,000
Invalid rows: 5,682

Top invalid reasons:
invalid_reason
street_only_no_geo_anchor    3992
missing_explicit_city        1488
province_city_conflict        100
city_not_resolved              98
ambiguous_area_multi_city       4

Pattern prevalence among invalid rows:
missing_city_word     93.8%
with_brgy_word         3.3%
street_only_tokens    64.8%
short_addr            46.8%
dtype: str

Sample invalid addresses (first 30):


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
4318,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",BLOCK 16 LOT 4 QUIOTAN STREET PHIL AM LIFE VIL...,NaN,NaN,NaN,street_only_no_geo_anchor
4319,2079 a elias st sta cruz,2079 A ELIAS STREET SANTA CRUZ,NaN,NaN,NaN,street_only_no_geo_anchor
4320,4662PagAsa St V mapa Sta Mesa Manila,4662PAGASA STREET V MAPA Santa Mesa MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4321,tayuman manila,TAYUMAN MANILA,NaN,NaN,NaN,missing_explicit_city
4322,CAGAYAN 3500 GOMEZ STREET,CAGAYAN 3500 GOMEZ STREET,NaN,NaN,NaN,street_only_no_geo_anchor
4323,873RizalStAlbayDistrict Bgy. 17 - Rizal Sreet....,873RIZALSTALBAYDISTRICT BARANGAY 17 RIZAL SREE...,CITY OF LEGAZPI,NaN,NaN,province_city_conflict
4324,"3101 DEEPWELL STOROSARIO,STA ROSA NUEVA ECIJA",3101 DEEPWELL STOROSARIO SANTA ROSA NUEVA ECIJA,CITY OF SANTA ROSA,NaN,NaN,province_city_conflict
4325,2161 Onyx St Santa Ana Manila,2161 ONYX STREET SANTA ANA MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4326,BLK 8 LOT 28 IBIZA TOWNHOMES GUINAYANG SAN MATEO,BLOCK 8 LOT 28 IBIZA TOWNHOMES GUINAYANG SAN M...,NaN,NaN,NaN,street_only_no_geo_anchor
4327,"RIZAL ST BRGY POBLACION VALLADOLID, NEGROS OCC",RIZAL STREET BARANGAY POBLACION MAKATI CITY VA...,VALLADOLID,NaN,NaN,province_city_conflict



Potentially recoverable invalids with city-like cues: 256


,original_address,address_normalized,city_signal,city_name,province_name,invalid_reason
4320,4662PagAsa St V mapa Sta Mesa Manila,4662PAGASA STREET V MAPA Santa Mesa MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4321,tayuman manila,TAYUMAN MANILA,NaN,NaN,NaN,missing_explicit_city
4325,2161 Onyx St Santa Ana Manila,2161 ONYX STREET SANTA ANA MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4327,"RIZAL ST BRGY POBLACION VALLADOLID, NEGROS OCC",RIZAL STREET BARANGAY POBLACION MAKATI CITY VA...,VALLADOLID,NaN,NaN,province_city_conflict
4344,room 323 calco bldg escolta st binindo manila,ROOM 323 CALCO BUILDING ESCOLTA STREET BININDO...,NaN,NaN,NaN,street_only_no_geo_anchor
4358,2211 KUSANG LOOB STREET SANTA CRUZ MANILA,2211 KUSANG LOOB STREET SANTA CRUZ MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4362,400 P CAPULONG STREET TONDO MANILA,400 P CAPULONG STREET TONDO I/II MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4365,CELADON PARK TOWER 3 STACRUZ MANILA,CELADON PARK TOWER 3 STACRUZ MANILA,NaN,NaN,NaN,street_only_no_geo_anchor
4367,1252 Carola StreetBrgy 489 Sampaloc1008 Manila,1252 CAROLA STREETBRGY 489 SAMPALOC1008 MANILA,NaN,NaN,NaN,missing_explicit_city
4370,ust sanpaloc manila,UST SANPALOC MANILA,NaN,NaN,NaN,missing_explicit_city


In [363]:
# Cell 14 -- Recoverable invalids snapshot (plain text)
cand = out_df[
    (~out_df['is_valid']) &
    out_df['address_normalized'].fillna('').astype(str).str.contains(
        r'\b(?:PQUE|PARANAQUE|PARAÑAQUE|QC|QUEZON|MANILA|MKT|MAKATI|TAGUIG|PASIG|MARIKINA|CEBU|DAVAO|CDO|CAGAYAN DE ORO)\b',
        regex=True, na=False
    )
].copy()

print(f'Recoverable candidates: {len(cand):,}')
cols = ['original_address', 'address_normalized', 'city_signal', 'city_name', 'province_name', 'invalid_reason']
for i, row in cand[cols].head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

print('\nCity-not-resolved rows:')
unres = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols]
for i, row in unres.head(60).iterrows():
    print('-' * 100)
    print(f"ORIG : {row['original_address']}")
    print(f"NORM : {row['address_normalized']}")
    print(f"SIG  : {row['city_signal']} | CITY: {row['city_name']} | PROV: {row['province_name']} | REASON: {row['invalid_reason']}")

Recoverable candidates: 268
----------------------------------------------------------------------------------------------------
ORIG : 4662PagAsa St V mapa Sta Mesa Manila
NORM : 4662PAGASA STREET V MAPA Santa Mesa MANILA
SIG  : nan | CITY: nan | PROV: nan | REASON: street_only_no_geo_anchor
----------------------------------------------------------------------------------------------------
ORIG : tayuman manila
NORM : TAYUMAN MANILA
SIG  : nan | CITY: nan | PROV: nan | REASON: missing_explicit_city
----------------------------------------------------------------------------------------------------
ORIG : 2161 Onyx St Santa Ana Manila
NORM : 2161 ONYX STREET SANTA ANA MANILA
SIG  : nan | CITY: nan | PROV: nan | REASON: street_only_no_geo_anchor
----------------------------------------------------------------------------------------------------
ORIG : RIZAL ST BRGY POBLACION VALLADOLID, NEGROS OCC
NORM : RIZAL STREET BARANGAY POBLACION MAKATI CITY VALLADOLID NEGROS OCC
SIG  : VALLADOLI

In [364]:
# Cell 15 -- Compact unresolved diagnostics
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

unres = tmp[tmp['invalid_reason'] == 'city_not_resolved'].copy()
print(f"city_not_resolved rows: {len(unres):,}")
for s in unres['norm'].head(20).tolist():
    print(' -', s[:140])

# Count metro-manila style abbreviations still not resolved
abbr_patterns = {
    'PQUE': r'\bPQUE\b',
    'MKT': r'\bMKT\b',
    'QC': r'\bQC\b',
    'MNL': r'\bMNL\b',
    'BGC': r'\bBGC\b',
    'CDO': r'\bCDO\b',
}
print('\nUnresolved abbreviation counts:')
for k, pat in abbr_patterns.items():
    c = int((~tmp['is_valid'] & tmp['norm'].str.contains(pat, regex=True, na=False)).sum())
    print(f' {k}: {c}')

city_not_resolved rows: 98
 - ILAYA LAS PIÑAS CITY
 - 15 INES MANAPUL BF RESORT CITY OF LAS PI?AS
 - 930 ROSAL STREET GATCHALIAN SUDB MANUYO CITY OF LAS PI?AS
 - LAS PIÑAS CITY CASIMIRO
 - CASA DE SEONORA CONDO LAS PIÑAS CITY
 - BF ALMANZA LAS PIÑAS CITY
 - BLOCK 5 LOT 8 DALANDAN STREET TALON 1 CITY OF LAS PI?AS
 - 26 EVELYN CAMUS STREET BF RESORT LAS PIÑAS CITY
 - 82 SAMPAGUITA STREET CAA CITY OF LAS PI?AS
 - BLOCK 61 LOT 87 PH 3 ALTA TIERRA HOMES BARANGAY A OLAES GENERAL MARIANO ALVAREZ CITY CAVITE
 - 20 IPIL CITY STREET NOVATIERRA VILLAGE
 - CITY OF LAS PI?AS HOME CASIMIRO VILLE
 - BLK4 LOT 11 DOMINIC HOMES E ALDANA CITY OF LAS PI?AS
 - 342 F LOZADA STREET PHASE 3 GATCHALIAN SUBDIVISION MANUYO 2 CITY OF LAS PI?AS
 - BLOCK 9 LOT 7 PNB HOMES AUGUSTO BARCELON BF RESORT CITY OF LAS PI?AS
 - LAS PIÑAS CITY PULANG LUPA
 - BLOCK 2 LOT 10 APARTMENT C GALASYA STREET CENTRAZA VILLAGE LAS PIÑAS CITY
 - LAS PIÑAS CITY CTIY
 - BLOCK 1 LOT 18 7A GARNET STREET MANUELA HOMES CITY OF LAS PI?AS
 - 21

In [365]:
# Cell 16 -- Unresolved detail view (plain text sample)
cols = ['original_address', 'address_normalized', 'city_signal', 'province_name', 'invalid_reason']
u = out_df[out_df['invalid_reason'] == 'city_not_resolved'][cols].copy()
print(f'city_not_resolved count: {len(u):,}')
for i, row in u.head(25).iterrows():
    print('-' * 90)
    print('ORIG :', str(row['original_address'])[:150])
    print('NORM :', str(row['address_normalized'])[:150])
    print('SIG  :', row['city_signal'], '| PROV:', row['province_name'], '| REASON:', row['invalid_reason'])

city_not_resolved count: 98
------------------------------------------------------------------------------------------
ORIG : Ilaya las pinas
NORM : ILAYA LAS PIÑAS CITY
SIG  : CITY OF LAS PI?AS | PROV: nan | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : 15 INES MANAPUL BF RESORT LAS PINAS CITY
NORM : 15 INES MANAPUL BF RESORT CITY OF LAS PI?AS
SIG  : CITY OF LAS PI?AS | PROV: nan | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : 930 Rosal St Gatchalian Sudb Manuyo Dos Las Pinas City
NORM : 930 ROSAL STREET GATCHALIAN SUDB MANUYO CITY OF LAS PI?AS
SIG  : CITY OF LAS PI?AS | PROV: nan | REASON: city_not_resolved
------------------------------------------------------------------------------------------
ORIG : LAS PINAS CASIMIRO
NORM : LAS PIÑAS CITY CASIMIRO
SIG  : CITY OF LAS PI?AS | PROV: nan | REASON: city_not_resolved
--------------

In [366]:
# Cell 17 -- Focus checks (La Carlota and Tondo)
tmp = out_df.copy()
tmp['norm'] = tmp['address_normalized'].fillna('').astype(str)

la = tmp[tmp['city_name'].fillna('').str.contains('LA CARLOTA', case=False, na=False)].copy()
la_false = la[~la['norm'].str.contains('LA CARLOTA', case=False, na=False)]
print(f'Rows with city_name=LA CARLOTA*: {len(la):,}')
print(f'LA CARLOTA assigned without mention: {len(la_false):,}')

tondo_rows = tmp[tmp['norm'].str.contains(r'\bTONDO\b', case=False, regex=True, na=False)].copy()
print(f'Rows mentioning TONDO: {len(tondo_rows):,}')
if len(tondo_rows):
    print('TONDO status counts:')
    print(tondo_rows['is_valid'].value_counts().to_string())

Rows with city_name=LA CARLOTA*: 1
LA CARLOTA assigned without mention: 0
Rows mentioning TONDO: 25
TONDO status counts:
is_valid
False    16
True      9


In [367]:
# Cell 18 -- Manila district city-key check from dim_location city_name
manila_cores = [
    'TONDO', 'BINONDO', 'ERMITA', 'INTRAMUROS', 'MALATE', 'PACO',
    'PANDACAN', 'PORT AREA', 'QUIAPO', 'SAMPALOC', 'SAN NICOLAS',
    'SANTA ANA', 'SANTA CRUZ',
]

def _canon_tmp(s: str) -> str:
    return re.sub(r'\s+', ' ', re.sub(r'[^A-Z0-9]+', ' ', str(s).upper())).strip()

canon_cities = [_canon_tmp(c) for c in cities]
print('Has CITY OF MANILA:', 'CITY OF MANILA' in canon_cities)
print('Has MANILA:', 'MANILA' in canon_cities)

print('\nDistrict core coverage in city_name:')
for core in manila_cores:
    ccore = _canon_tmp(core)
    hits = sorted({cities[i] for i, c in enumerate(canon_cities) if f' {ccore} ' in f' {c} '})
    print(f'[{core}] -> {len(hits)} match(es)')
    for h in hits[:8]:
        print('  -', h)

Has CITY OF MANILA: False
Has MANILA: False

District core coverage in city_name:
[TONDO] -> 1 match(es)
  - TONDO I/II
[BINONDO] -> 1 match(es)
  - BINONDO
[ERMITA] -> 1 match(es)
  - ERMITA
[INTRAMUROS] -> 1 match(es)
  - INTRAMUROS
[MALATE] -> 1 match(es)
  - MALATE
[PACO] -> 1 match(es)
  - PACO
[PANDACAN] -> 1 match(es)
  - PANDACAN
[PORT AREA] -> 1 match(es)
  - PORT AREA
[QUIAPO] -> 1 match(es)
  - QUIAPO
[SAMPALOC] -> 1 match(es)
  - SAMPALOC
[SAN NICOLAS] -> 1 match(es)
  - SAN NICOLAS
[SANTA ANA] -> 1 match(es)
  - SANTA ANA
[SANTA CRUZ] -> 1 match(es)
  - SANTA CRUZ


In [368]:
# Cell 19 -- Tondo mapping probe in dim_location
probe = dim[dim['barangay_name'].fillna('').astype(str).str.contains('TONDO', case=False, na=False)].copy()
print(f'Rows with barangay containing TONDO: {len(probe):,}')
if len(probe):
    print('Unique city names:')
    for c in sorted(probe['city_name'].dropna().unique().tolist())[:20]:
        print(' -', c)
    print('Unique province names:')
    for p in sorted(probe['province_name'].dropna().unique().tolist())[:10]:
        print(' -', p)

Rows with barangay containing TONDO: 6
Unique city names:
 - ANDA
 - CABANGAN
 - CLAVER
 - SAN JOSE CITY
 - SAN JUAN
 - TANGALAN
Unique province names:
 - AKLAN
 - LA UNION
 - NUEVA ECIJA
 - PANGASINAN
 - SURIGAO DEL NORTE
 - ZAMBALES


In [369]:
# Targeted Manila district sanity check
for raw in ['TONDO MANILA', 'BINONDO MANILA', 'ERMITA MANILA', 'INTRAMUROS MANILA']:
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(raw, '=>', ans['city_signal'], '|', ans['city_name'], '|', ans['invalid_reason'])

TONDO MANILA => TONDO I/II | TONDO I/II | None
BINONDO MANILA => BINONDO | BINONDO | None
ERMITA MANILA => ERMITA | ERMITA | None
INTRAMUROS MANILA => INTRAMUROS | INTRAMUROS | None


## Cell 6 — Load to API and Map Back to PSGC

Send unresolved or low-confidence rows to OSM Nominatim, then map the returned city, province, and region back to `dim_location` to recover PSGC codes.

In [370]:
# Compatibility bootstrap for the legacy "IMPROVED PIPELINE" cells below.
# It maps current notebook variables to the names expected by that section.

def normalize_text(text):
    import re, pandas as pd
    if pd.isna(text):
        return ''
    text = text.upper()
    text = re.sub(r'LANDMARK.*', '', text)
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Ensure df exists with Original_Address column.
if 'df' not in globals():
    if 'out_df' in globals() and isinstance(out_df, pd.DataFrame) and not out_df.empty:
        df = out_df.copy()
    elif 'df_input' in globals() and isinstance(df_input, pd.DataFrame) and not df_input.empty:
        df = df_input.copy()
    else:
        df = pd.read_excel(INPUT_FILE)

if 'Original_Address' not in df.columns:
    if 'original_address' in df.columns:
        df['Original_Address'] = df['original_address']
    else:
        df['Original_Address'] = df.iloc[:, 0].astype(str)

# Build legacy alias schema expected by downstream cells.
if 'alias' not in globals():
    if 'alias_df' in globals() and {'pattern', 'replacement'}.issubset(alias_df.columns):
        alias = alias_df[['pattern', 'replacement']].copy()
        alias = alias.rename(columns={'pattern': 'alias', 'replacement': 'standard_name'})
    else:
        alias = pd.DataFrame(columns=['alias', 'standard_name'])

# Build legacy dim_location schema expected by downstream cells.
if 'dim_location' not in globals():
    if 'dim' in globals() and {'barangay_name', 'city_name'}.issubset(dim.columns):
        dim_location = dim.copy()
        rename_map = {
            'barangay_name': 'barangay',
            'city_name': 'city',
            'province_name': 'province',
            'region_name': 'region',
        }
        dim_location = dim_location.rename(columns=rename_map)
    else:
        dim_location = pd.DataFrame(columns=['barangay', 'city', 'province', 'region'])

print(f'df rows: {len(df):,} | alias rows: {len(alias):,} | dim_location rows: {len(dim_location):,}')

df rows: 100 | alias rows: 514 | dim_location rows: 42,011


In [371]:

import re

def extract_components(text):
    result = {"barangay": None, "city": None}

    brgy_match = re.search(r'(BRGY|BARANGAY)?\s*([A-Z\-\s]{3,})', text)
    if brgy_match:
        result["barangay"] = brgy_match.group(2).strip()

    city_patterns = [
        r'QUEZON CITY', r'MANILA', r'BAGUIO', r'LAS\s*PINAS', r'DAVAO', r'CEBU'
    ]

    for pattern in city_patterns:
        if re.search(pattern, text):
            result["city"] = pattern.replace("\\s*", " ")

    return result


In [372]:

df['clean_address'] = df['Original_Address'].apply(normalize_text)

components = df['clean_address'].apply(extract_components)
df['brgy_candidate'] = components.apply(lambda x: x['barangay'])
df['city_candidate'] = components.apply(lambda x: x['city'])


In [373]:

alias_dict = dict(zip(alias['alias'].str.upper(), alias['standard_name'].str.upper()))

def apply_alias(val):
    if pd.isna(val):
        return None
    return alias_dict.get(val.upper(), val)

df['brgy_norm'] = df['brgy_candidate'].apply(apply_alias)
df['city_norm'] = df['city_candidate'].apply(apply_alias)


In [374]:
from rapidfuzz import fuzz, process

barangay_list = dim_location['barangay'].dropna().unique()

def match_address_row(row):
    import pandas as pd

    brgy = row['brgy_norm']
    city = row['city_norm']

    score = 0
    match = None

    if pd.notna(brgy):
        res = process.extractOne(brgy, barangay_list, scorer=fuzz.ratio)
        if res:
            match_name, s, _ = res
            if s >= 85:
                score += 40
                match = match_name

    candidates = dim_location.copy()

    if match is not None:
        candidates = candidates[candidates['barangay'] == match]

    if pd.notna(city):
        candidates = candidates[candidates['city'].str.contains(city, na=False)]
        score += 40

    if len(candidates) > 1:
        return pd.Series([None, "AMBIGUOUS", score])

    if len(candidates) == 1 and score >= 70:
        return pd.Series([candidates.iloc[0]['barangay'], "MATCHED", score])

    return pd.Series([None, "LOW_CONFIDENCE", score])
